# ChestXRay Diagnosis Tool — Data Preprocessing

This notebook downloads the NIH Chest X-ray 14 dataset from Kaggle,
filters it down to our **3 target diseases** + healthy baseline,
and packages a clean, labelled dataset for training.

| Disease | NIH CSV Label |
|---|---|
| Cardiomegaly (Enlarged Heart) | `Cardiomegaly` |
| Pleural Effusion (Fluid in Lungs) | `Effusion` |
| Pneumothorax (Collapsed Lung) | `Pneumothorax` |
| Healthy Baseline | `No Finding` |

## Step 1 — Kaggle Setup & Download the CSV

In [ ]:
!pip install -q kaggle

# Upload your kaggle.json when prompted
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download JUST the CSV first (tiny ~1 MB)
!kaggle datasets download -d nih-chest-xrays/data -f Data_Entry_2017.csv
!unzip -o Data_Entry_2017.csv.zip

## Step 2 — Filter the CSV for our 3 target diseases

**Important:** The NIH CSV uses `"Effusion"` — NOT `"Pleural_Effusion"`.
We use **set-based matching** (not regex) to avoid false positives like `Pleural_Thickening`.

In [ ]:
import pandas as pd

df = pd.read_csv('Data_Entry_2017.csv')
print(f"Total images in dataset: {len(df):,}")

# --- Target diseases (EXACT label strings from the NIH CSV) ---
TARGET_DISEASES = ['Cardiomegaly', 'Effusion', 'Pneumothorax']
NO_FINDING = 'No Finding'

# Parse pipe-separated labels into a Python set per row
# e.g. "Cardiomegaly|Effusion" -> {'Cardiomegaly', 'Effusion'}
df['labels_set'] = df['Finding Labels'].apply(lambda s: set(str(s).split('|')))

# Keep rows with at least one target disease OR pure "No Finding"
targets = set(TARGET_DISEASES)
mask = df['labels_set'].apply(
    lambda s: bool(s & targets) or s == {NO_FINDING}
)
filtered_df = df[mask].copy()

# --- Multi-hot encoded label columns ---
for disease in TARGET_DISEASES:
    filtered_df[f'label_{disease}'] = filtered_df['labels_set'].apply(
        lambda s, d=disease: int(d in s)
    )

filtered_df.drop(columns=['labels_set'], inplace=True)

# --- Stats ---
images_to_keep = set(filtered_df['Image Index'].tolist())
print(f"\nFiltered to {len(filtered_df):,} images.\n")
print("Label distribution:")
for disease in TARGET_DISEASES:
    col = f'label_{disease}'
    pos = filtered_df[col].sum()
    print(f"  {disease:<20} {pos:>6,} positives  ({pos/len(filtered_df)*100:.1f}%)")

no_finding = (filtered_df[[f'label_{d}' for d in TARGET_DISEASES]].sum(axis=1) == 0).sum()
print(f"  {'No Finding':<20} {no_finding:>6,} images    ({no_finding/len(filtered_df)*100:.1f}%)")

## Step 3 — Download & extract the full image dataset

This downloads ~45 GB from Kaggle to Colab's cloud workspace (cloud-to-cloud, ~1-2 min).  
Then unzips all 112K images.

In [ ]:
# Download the full dataset (cloud-to-cloud, very fast)
!kaggle datasets download -d nih-chest-xrays/data

# Unzip everything into "all_images/"
!unzip -q data.zip -d all_images

# Verify the sub-folder structure
!ls all_images/

## Step 4 — Copy only our filtered images into a clean folder

The NIH dataset extracts into **12 sub-folders** (`images_001/` through `images_012/`),  
each containing an `images/` subdirectory. We scan **all** of them using `rglob`.

In [ ]:
import os
import shutil
from pathlib import Path

SRC_ROOT = Path('all_images')
DST_DIR = Path('my_custom_dataset')
DST_DIR.mkdir(exist_ok=True)

# Build a lookup: filename -> full path (scans all 12 sub-folders)
print('Building image index across all sub-folders...')
image_index = {}
for png in SRC_ROOT.rglob('*.png'):
    image_index[png.name] = png
print(f'Found {len(image_index):,} total .png files.\n')

# Copy only the filtered images
copied = 0
missing = []
for i, filename in enumerate(images_to_keep):
    src = image_index.get(filename)
    if src:
        shutil.copy2(src, DST_DIR / filename)
        copied += 1
    else:
        missing.append(filename)

    if (i + 1) % 5000 == 0:
        print(f'  Copied {copied:,} / {i+1:,} ...')

print(f'\nDone! Copied {copied:,} images to {DST_DIR}/')
if missing:
    print(f'Warning: {len(missing)} images not found (possibly corrupt/missing).')

## Step 5 — Save the filtered CSV alongside the images

This CSV contains filenames + multi-hot labels — everything needed for training.

In [ ]:
csv_out = 'my_custom_dataset/labels.csv'
filtered_df.to_csv(csv_out, index=False)
print(f'Saved filtered CSV -> {csv_out}')
print(f'Columns: {list(filtered_df.columns)}')
filtered_df.head()

## Step 6 — Zip & download your custom dataset

Compress just the filtered images + CSV into a single zip.  
This will be a **fraction** of the original 45 GB.

In [ ]:
!zip -r -q my_custom_dataset.zip my_custom_dataset/

import os
size_gb = os.path.getsize('my_custom_dataset.zip') / (1024**3)
print(f'\nmy_custom_dataset.zip  ->  {size_gb:.2f} GB')
print(f'(vs ~45 GB for the full dataset)\n')

# Download to your laptop
from google.colab import files
files.download('my_custom_dataset.zip')